# Fine-tune Qwen2.5-3B-Instruct bằng Unsloth (TinyLoRA) trên VlogQA
Notebook này fine-tune **Qwen2.5-3B-Instruct** với **TinyLoRA** — phiên bản nhẹ hơn LoRA tiêu chuẩn:

| Tham số | LoRA chuẩn | **TinyLoRA** |
|---|---|---|
| Rank `r` | 16 | **8** |
| `lora_alpha` | 32 | **16** |
| Quantization | BF16 (full) | **4-bit NF4 (QLoRA)** |
| `max_seq_length` | 8192 | **4096** |
| `gradient_accumulation_steps` | 8 | **4** |

**Mục đích:** Tiết kiệm VRAM ~40-50%, phù hợp GPU VRAM ≤ 8GB.

VlogQA là dataset **Extractive QA** (dạng SQuAD): mô hình trích xuất span từ context.

**Evaluation metrics:** Exact Match (EM) và F1-score.

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [1]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 11.3 MB/s  0:00:58:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 11.3 MB/s  0:00:25:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 11.2 MB/s  0:00:26:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 11.5 MB/s  0:00:12:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 11.4 MB/s  0:00:52:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 11.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 11.3 MB/s  0:00:17:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 11.0 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 11.7 MB/s  0:00:05m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 M

In [3]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-k5le7ten/unsloth_9a6dfd6e8e854a9bbb7fdca803abe1db
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-k5le7ten/unsloth_9a6dfd6e8e854a9bbb7fdca803abe1db
  Resolved https://github.com/unslothai/unsloth.git to commit 1777aae37ee8ecd6e5ba5b303f4505d975382751
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## 1. Load Model và Tokenizer với Unsloth (QLoRA 4-bit)

**TinyLoRA** nạp model ở chế độ **4-bit NF4 quantization** để giảm VRAM ~50% so với BF16.

In [1]:
from unsloth import FastLanguageModel
import torch

# ==========================================
# CẤU HÌNH TINYLORA — nhẹ hơn LoRA chuẩn
# ==========================================
max_seq_length = 4096  # Giảm từ 8192 → 4096: tiết kiệm VRAM
dtype = None           # None để auto-detect (BFloat16 nếu GPU hỗ trợ)
load_in_4bit = False    # QLoRA: nạp model 4-bit NF4 (~2GB VRAM thay vì ~6GB)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Tải model thành công! (QLoRA 4-bit)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/unsloth_zoo/_vendored/fla/utils/_device.py:100: UserWarning: Triton is not supported on current platform, roll back to CPU.
  _cpu_device_warning()
/opt/venv/lib/python3.12/site-packages/unsloth_zoo/_vendored/fla/utils/_device.py:165: UserWarning: Triton is not supported on current platform, roll back to CPU.
  _cpu_device_warning()
/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.682 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Tải model thành công! (QLoRA 4-bit)


## 2. Cấu hình TinyLoRA Adapter

**TinyLoRA** dùng rank **r=8** và **alpha=16** (nhỏ hơn LoRA chuẩn r=16/alpha=32).
Số trainable parameters giảm 4×, tốc độ huấn luyện nhanh hơn và ít overfitting hơn trên tập nhỏ.

In [ ]:
from peft import TinyLoraConfig, get_peft_model, prepare_model_for_kbit_training
import peft

# 1. Chuẩn bị model cho 4-bit training (BẮT BUỘC nếu load_in_4bit=True)
# Giúp cast các layer cần thiết sang fp32 để tính gradient
model = prepare_model_for_kbit_training(
    model, 
    use_gradient_checkpointing=True
)

# 2. Cấu hình TinyLoRA thực sự (True TinyLoRA)
tinylora_config = TinyLoraConfig(
    r = 2,
    u = 64,                     
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    tinylora_dropout = 0.0,     
    bias = "none",
    task_type = "CAUSAL_LM",
    weight_tying = 0.0,
    projection_seed = 3407,
    init_weights = True,
)

# 3. Gắn TinyLoRA adapter
model = get_peft_model(model, tinylora_config)
model.config.use_cache = False

# Ép kiểu các tham số trainable về float32 để tránh lỗi "Only Tensors of floating point..."
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

# In tóm tắt số lượng tham số
model.print_trainable_parameters()
print(f"Đã áp dụng cấu hình TinyLoRA thực thụ (PEFT {peft.__version__})!")


## 3. Chuẩn bị Dataset VlogQA

In [ ]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""
                if answer:
                    samples.append({
                        "context":  context,
                        "question": question,
                        "answer":   answer,
                        "id":       qa.get("id", ""),
                    })
    return samples


# ==========================================
# Anchor-based Context Truncation
# ==========================================
MAX_CONTEXT_TOKENS = 7500

def truncate_context_around_answer(context, answer, tokenizer, max_ctx_tokens=MAX_CONTEXT_TOKENS):
    answer_start = context.find(answer)
    if answer_start == -1:
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        if len(ctx_ids) <= max_ctx_tokens:
            return context
        return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    before_text = context[:answer_start]
    after_text  = context[answer_start + len(answer):]

    before_ids = tokenizer.encode(before_text, add_special_tokens=False)
    answer_ids = tokenizer.encode(answer,       add_special_tokens=False)
    after_ids  = tokenizer.encode(after_text,   add_special_tokens=False)

    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context

    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2

    before_keep = before_ids[-half:]
    after_keep  = after_ids[:budget - half]

    merged_ids = before_keep + answer_ids + after_keep
    return tokenizer.decode(merged_ids, skip_special_tokens=True)


# ==========================================
# System Prompt & User Template (đồng bộ Train/Eval)
# ==========================================
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm \'Đáp án:\' hay bất kỳ tiền tố nào.\n"
    "3) Câu trả lời phải là một chuỗi ký tự CÓ MẶT trong đoạn văn được cung cấp."
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Câu trả lời (span-only):"
)


def format_prompt_train(context, question, answer, tokenizer):
    """Tạo prompt đầy đủ (input + output) để huấn luyện."""
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context_cropped,
                question=question
            )
        },
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def format_prompt_inference(context, question, tokenizer):
    """Tạo prompt inference (chỉ input, không có answer)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": USER_PROMPT_TEMPLATE.format(
                context=context[:7500],
                question=question
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

TRAIN_DATA_PATH = "train.json"
DEV_DATA_PATH = "dev.json"  # Đường dẫn tới tập dev/val

train_samples = load_vlogqa(TRAIN_DATA_PATH)
eval_samples = load_vlogqa(DEV_DATA_PATH)

print(f"Số lượng mẫu Train: {len(train_samples)}")
print(f"Số lượng mẫu Eval: {len(eval_samples)}")

# ==========================================
# Chuyển thành HuggingFace Dataset
# ==========================================
def prepare_hf_dataset(samples, tokenizer):
    formatted = []
    for s in samples:
        text = format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

train_dataset = prepare_hf_dataset(train_samples, tokenizer)
eval_dataset  = prepare_hf_dataset(eval_samples, tokenizer)

print(f"\nDataset HuggingFace tạo thành công:")
print(f"Train: {len(train_dataset)} mẫu")
print(f"Eval: {len(eval_dataset)} mẫu")


## 4. Cấu hình và Bắt đầu Huấn luyện (TinyLoRA Fine-Tuning)

**Điểm khác biệt so với LoRA chuẩn:**
- `gradient_accumulation_steps = 4` (thay vì 8) → effective batch = 2×4 = 8
- `max_seq_length = 4096` → chuỗi ngắn hơn, nhanh hơn
- `output_dir = "outputs_vlogqa_tinylora"` → không ghi đè kết quả LoRA chuẩn

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported
import sys

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,          # 4096 (TinyLoRA)
    dataset_num_proc = 1,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,      # TinyLoRA: 4 (thay vì 8); Effective batch = 2×4 = 8
        warmup_steps = 30,                    # Ít warmup hơn vì effective batch nhỏ hơn
        num_train_epochs = 3,                 # Để 3 hoặc tăng lên 5 nếu dùng early stopping
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 100,                     # Đánh giá mỗi 100 steps
        save_strategy = "steps",
        save_steps = 100,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_vlogqa_tinylora",
        report_to = "none",
    ),
    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 2,      # Dừng nếu 2 lần eval liên tiếp không cải thiện
            early_stopping_threshold = 0.01
        )
    ]
)

# Fix sys.modules để tránh lỗi pickle với unsloth
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

print("\nBắt đầu TinyLoRA fine-tuning (có Early Stopping)...")
trainer_stats = trainer.train()


## 5. Kiểm tra Inference nhanh sau khi huấn luyện

In [ ]:
# Kích hoạt chế độ sinh token nhanh của Unsloth (2x tốc độ)
FastLanguageModel.for_inference(model)

# Thử nghiệm với mẫu đầu tiên trong tập train
sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 64,
    use_cache = True,
    do_sample = False,
    temperature = 1.0,
)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE (TinyLoRA) ---")
print(f"Câu hỏi    : {sample['question']}")
print(f"Đáp án đúng: {sample['answer']}")
print(f"Model trả lời: {response}")

## 6. Lưu Model TinyLoRA

In [ ]:
# Chỉ lưu TinyLoRA weights (nhẹ hơn LoRA chuẩn — rank nhỏ hơn)
LORA_SAVE_PATH = "qwen2.5-3b-instruct-tinylora-vlogqa"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình TinyLoRA tại: {LORA_SAVE_PATH}")
print(f"(Adapter nhỏ hơn LoRA chuẩn do rank r=8 thay vì r=16)")

# Tùy chọn: Merge TinyLoRA vào model gốc và lưu dạng full weights
# model.save_pretrained_merged("qwen2.5-3b-instruct-tinylora-vlogqa-merged", tokenizer, save_method="merged_16bit")

---
## 7. Đánh giá (Evaluation) trên tập Test
### Metrics: Exact Match (EM) và F1-score

- **Exact Match (EM):** Tỷ lệ câu trả lời của model khớp **hoàn toàn** với đáp án.
- **F1-score:** Đo lường overlap từ-vựng giữa dự đoán và đáp án.

> **Lưu ý:** Cell này có thể chạy trong một session mới sau khi đã train và lưu model.

In [1]:
import json
import string
import unicodedata
import torch
from tqdm.auto import tqdm
from collections import Counter

# ==========================================
# CẤU HÌNH EVAL
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH    = "qwen2.5-3b-instruct-tinylora-vlogqa"
TEST_DATA_PATH  = "test.json"
MAX_EVAL_TOKENS = 3500

SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện nguyên văn trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:' hay bất kỳ tiền tố nào.\n"
    "3) Câu trả lời phải là một chuỗi ký tự CÓ MẶT trong đoạn văn được cung cấp."
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    "Câu trả lời (span-only):"
)


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# LOAD MODEL ĐÃ FINE-TUNE (TinyLoRA)
# ==========================================
import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

os.environ["HF_HUB_OFFLINE"] = "1"
BASE_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"

print("Loading Tokenizer và Model TinyLoRA (bằng HF thuần)... ")
try:
    tokenizer_eval = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    model_eval = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="cuda"
    )
    model_eval = PeftModel.from_pretrained(model_eval, ADAPTER_PATH, is_trainable=False)
    model_eval.config.use_cache = True
    model_eval.eval()
    if tokenizer_eval.pad_token is None:
        tokenizer_eval.pad_token = tokenizer_eval.eos_token
    print("Load TinyLoRA model hoàn tất! (Sẵn sàng cho use_cache=True)")
except Exception as e:
    print(f"Lỗi: {e}")


Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /opt/venv/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Loading Tokenizer và Model TinyLoRA (bằng HF thuần)... 


`torch_dtype` is deprecated! Use `dtype` instead!



Load TinyLoRA model hoàn tất! (Sẵn sàng cho use_cache=True)


In [3]:
# ==========================================
# HÀM TIỆN ÍCH: CHUẨN HÓA VĂN BẢN
# ==========================================
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())
    return text


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)


def get_best_score(prediction: str, answers: list) -> tuple:
    em_scores = [compute_exact_match(prediction, ans["text"]) for ans in answers]
    f1_scores = [compute_f1_token(prediction, ans["text"]) for ans in answers]
    return max(em_scores), max(f1_scores)


# ==========================================
# Best-Span Post-processing
# ==========================================
def find_best_span(prediction: str, context: str) -> str:
    """
    Sau khi model sinh ra text, tìm substring trong context
    có token-F1 cao nhất so với prediction.
    """
    pred_norm = normalize_text(prediction)
    if not pred_norm:
        return prediction

    pred_words = pred_norm.split()
    ctx_norm   = normalize_text(context)
    ctx_words  = ctx_norm.split()
    ctx_orig   = context.split()

    n = len(pred_words)
    if n == 0 or len(ctx_words) == 0:
        return prediction

    baseline_f1 = compute_f1_token(pred_norm, ctx_norm)
    best_f1     = baseline_f1
    best_span   = prediction

    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))

    for w in range(min_w, max_w + 1):
        for i in range(len(ctx_words) - w + 1):
            span_norm = " ".join(ctx_words[i:i + w])
            f1 = compute_f1_token(pred_norm, span_norm)
            if f1 > best_f1:
                best_f1   = f1
                best_span = " ".join(ctx_orig[i:i + w])

    return best_span


# ==========================================
# HÀM TẠO PROMPT INFERENCE (có pre-truncate)
# ==========================================
def truncate_context_for_eval(context, tokenizer, max_ctx_tokens=MAX_EVAL_TOKENS):
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) <= max_ctx_tokens:
        return context
    return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)


def format_prompt_inference_eval(context, question, tokenizer):
    context_cropped = truncate_context_for_eval(context, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đoạn văn:\n{context_cropped}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Trích xuất nguyên văn từ đoạn văn (không thêm bớt):"
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")
print("Các hàm metric và best-span đã sẵn sàng!")

Tổng số câu hỏi test: 1062
Các hàm metric và best-span đã sẵn sàng!


In [6]:
# ==========================================
# CHẠY BATCH INFERENCE TRÊN TOÀN BỘ TẬP TEST
# ==========================================
import time
import warnings
warnings.filterwarnings("ignore", message="Both `max_new_tokens`")

# ---- Cấu hình Batch ----
BATCH_SIZE    = 8
SUMMARY_EVERY = 100

all_em_raw      = []
all_f1_raw      = []
all_em_span     = []
all_f1_span     = []
all_predictions = []

TOTAL      = len(test_samples)
n_batches  = (TOTAL + BATCH_SIZE - 1) // BATCH_SIZE
start_time = time.time()

print(f"\nBắt đầu đánh giá trên {TOTAL} câu hỏi...")
print(f"Batch size = {BATCH_SIZE} | Số batch = {n_batches} | use_cache=True\n")

pbar = tqdm(range(0, TOTAL, BATCH_SIZE), desc="Batch Evaluating", total=n_batches)

for batch_start in pbar:
    batch = test_samples[batch_start : batch_start + BATCH_SIZE]

    # Tạo prompt cho từng sample trong batch
    prompts = [
        format_prompt_inference_eval(
            context   = s["context"],
            question  = s["question"],
            tokenizer = tokenizer_eval
        )
        for s in batch
    ]

    # Tokenize cả batch (left-padding, cắt tối đa 4096)
    inputs = tokenizer_eval(
        prompts,
        return_tensors = "pt",
        padding        = True,
        truncation     = True,
        max_length     = 4096,
    ).to(model_eval.device)

    total_input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_new_tokens = 64,
            do_sample      = False,
            temperature    = 1.0,
            use_cache      = True,
            pad_token_id   = tokenizer_eval.pad_token_id,
            eos_token_id   = tokenizer_eval.eos_token_id,
        )

    # Decode từng sample trong batch
    for j, sample in enumerate(batch):
        gen_tokens     = outputs[j][total_input_len:]
        raw_prediction = tokenizer_eval.decode(gen_tokens, skip_special_tokens=True).strip()
        raw_prediction = raw_prediction.split("\n")[0].strip()

        # Dùng lại hàm xử lý nhãn (nếu import_clean_prediction chưa định nghĩa thì dùng hàm mặc định)
        span_prediction = find_best_span(raw_prediction, sample["context"])

        # Tính metric
        em_raw,  f1_raw  = get_best_score(raw_prediction, sample["answers"])
        em_span, f1_span = get_best_score(span_prediction,    sample["answers"])

        all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
        all_em_span.append(em_span); all_f1_span.append(f1_span)
        all_predictions.append({
            "id":               sample["id"],
            "question":         sample["question"],
            "ground_truth":     sample["answers"][0]["text"],
            "raw_prediction":   raw_prediction,
            "span_prediction":  span_prediction,
            "em_raw":  em_raw,  "f1_raw":  f1_raw,
            "em_span": em_span, "f1_span": f1_span,
        })

        idx = len(all_predictions)

        # In ra sample đầu tiên của mỗi batch hoặc 10 câu đầu để test
        if idx <= 10 or (idx % BATCH_SIZE == 0 and idx <= SUMMARY_EVERY):
            tqdm.write(
                f"[{idx}/{TOTAL}] "
                f"Q: {sample['question'][:40]}... | "
                f"Truth: {sample['answers'][0]['text'][:30]} | "
                f"Raw: {raw_prediction[:30]} | "
                f"Span: {span_prediction[:30]} | "
                f"EM_raw={em_raw} EM_span={em_span} "
                f"F1_raw={f1_raw:.3f} F1_span={f1_span:.3f}"
            )

    # Cập nhật tqdm postfix sau mỗi batch
    idx     = len(all_predictions)
    elapsed = time.time() - start_time
    eta     = (elapsed / idx) * (TOTAL - idx) if idx > 0 else 0
    cur_em  = sum(all_em_span) / idx * 100
    cur_f1  = sum(all_f1_span) / idx * 100
    pbar.set_postfix({"EM": f"{cur_em:.1f}%", "F1": f"{cur_f1:.1f}%", "ETA": f"{eta/60:.1f}m"})

    # Tóm tắt mỗi SUMMARY_EVERY câu
    if idx % SUMMARY_EVERY < BATCH_SIZE and idx >= SUMMARY_EVERY:
        tqdm.write(f"\n{'='*60}")
        tqdm.write(f"  [Checkpoint ~{idx}/{TOTAL}] Đã chạy {elapsed:.0f}s | ETA ~{eta/60:.1f} phút")
        tqdm.write(f"  EM  (raw):  {sum(all_em_raw)/idx*100:.2f}%  |  EM  (span): {cur_em:.2f}%")
        tqdm.write(f"  F1  (raw):  {sum(all_f1_raw)/idx*100:.2f}%  |  F1  (span): {cur_f1:.2f}%")
        tqdm.write(f"{'='*60}\n")

total_time = time.time() - start_time
print(f"\nHoàn tất inference! Tổng thời gian: {total_time/60:.1f} phút")
print(f"Tốc độ trung bình: {TOTAL/total_time:.1f} câu/giây")

n = TOTAL
print("\n" + "="*60)
print("KẾT QUẢ ĐÁNH GIÁ TỔNG THỂ — TinyLoRA (Qwen2.5-3B-Instruct)")
print("="*60)
print(f"{'Metric':<25} {'Raw Output':>12} {'Best-Span':>12}")
print("-"*50)
print(f"{'Exact Match (EM)':<25} {sum(all_em_raw)/n*100:>11.2f}% {sum(all_em_span)/n*100:>11.2f}%")
print(f"{'F1-score':<25} {sum(all_f1_raw)/n*100:>11.2f}% {sum(all_f1_span)/n*100:>11.2f}%")
print("-"*50)


Bắt đầu đánh giá trên 1062 câu hỏi...
Batch size = 8 | Số batch = 133 | use_cache=True



Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/1062] Q: Lò được làm nóng trước khi nướng bao lâu... | Truth: 10 phút | Raw: 10分钟 | Span: 10分钟 | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000
[2/1062] Q: Đầu bếp sử dụng gì để trang trí bánh?... | Truth: mấy cái hoa hồi | Raw: hoa hồi | Span: hoa hồi | EM_raw=0 EM_span=0 F1_raw=0.667 F1_span=0.667
[3/1062] Q: Bột ca cao được hòa tan bằng gì?... | Truth: nước ấm | Raw: ca cao ca cao ca cao cacacacac | Span: bột ca cao | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000
[4/1062] Q: Việc làm nóng lò trước khi nướng để làm ... | Truth: tạo rễ tre cho bánh và để bánh | Raw: Làm nóng lò trước khi nướng để | Span: để tạo rễ tre cho bánh và để b | EM_raw=0 EM_span=0 F1_raw=0.759 F1_span=0.957
[5/1062] Q: Nếu để hỗn hợp ca cao sôi mạnh thì sẽ mấ... | Truth: nước cốt dừa | Raw: nước cốt dừa | Span: nước cốt dừa | EM_raw=1 EM_span=1 F1_raw=1.000 F1_span=1.000
[6/1062] Q: Nếu sử dụng trứng vịt thì bánh flan sẽ n... | Truth: ngon hơn | Raw: Rụng trứng vịt làm bánh flan t | Span: rụng trứng vịt nó s

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[8/1062] Q: Dùng cái gì sâm vào bánh để kiểm tra bán... | Truth: cái que | Raw: sâm một cái que | Span: sâm một cái que | EM_raw=0 EM_span=0 F1_raw=0.667 F1_span=0.667
[9/1062] Q: Đầu bếp đã sử dụng bao nhiêu đường thốt ... | Truth: 100g | Raw: 50ml | Span: 50ml | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000
[10/1062] Q: Mặc dù có trứng nhưng trong bánh không c... | Truth: tanh mùi trứng | Raw: mát luôn | Span: luôn | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000


...


  [Checkpoint ~904/1062] Đã chạy 5355s | ETA ~15.6 phút
  EM  (raw):  15.93%  |  EM  (span): 16.15%
  F1  (raw):  34.87%  |  F1  (span): 33.45%



Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


  [Checkpoint ~1000/1062] Đã chạy 5942s | ETA ~6.1 phút
  EM  (raw):  15.70%  |  EM  (span): 15.60%
  F1  (raw):  34.68%  |  F1  (span): 33.09%



Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


Hoàn tất inference! Tổng thời gian: 102.0 phút
Tốc độ trung bình: 0.2 câu/giây

KẾT QUẢ ĐÁNH GIÁ TỔNG THỂ — TinyLoRA (Qwen2.5-3B-Instruct)
Metric                      Raw Output    Best-Span
--------------------------------------------------
Exact Match (EM)                15.63%       15.73%
F1-score                        34.70%       33.22%
--------------------------------------------------


In [8]:
# ==========================================
# LƯU KẾT QUẢ ĐÁNH GIÁ (PREDICTIONS & TỔNG ĐIỂM)
# ==========================================
import json

OUTPUT_FILE = "tinylora_vlogqa_test_results.json"

n = len(all_predictions)
final_results = {
    "summary": {
        "total_samples": n,
        "em_raw_percent": sum(all_em_raw) / n * 100 if n > 0 else 0,
        "f1_raw_percent": sum(all_f1_raw) / n * 100 if n > 0 else 0,
        "em_span_percent": sum(all_em_span) / n * 100 if n > 0 else 0,
        "f1_span_percent": sum(all_f1_span) / n * 100 if n > 0 else 0
    },
    "details": all_predictions
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_results, f, ensure_ascii=False, indent=4)

print(f"\nĐã lưu BẢNG ĐIỂM TỔNG và {n} kết quả chi tiết vào file: {OUTPUT_FILE}")



Đã lưu BẢNG ĐIỂM TỔNG và 1062 kết quả chi tiết vào file: tinylora_vlogqa_test_results.json
